In [1]:
pip install huggingface_hub[hf_xet]

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------- ----------------------------- 0.8/2.9 MB 4.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.8/2.9 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------  2.9/2.9 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 4.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

app = FastAPI()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

class BotRequest(BaseModel):
    question: str

@app.post("/api/generate-bot-answer")
def generate_bot_answer(body: BotRequest):
    prompt = (
        "You are a human player in a party game called 'Spot the Bot'. "
        "You must answer the question like a normal person, not like an AI. "
        "Be short, casual, and a bit messy if needed.\n\n"
        f"Question: {body.question}\n"
        "Your answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.9,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    full = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Strip prompt prefix if it appears in output
    if full.startswith(prompt):
        full = full[len(prompt):]

    answer = full.strip()
    return {"answer": answer}


c:\My files\Programming\globvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\My files\Programming\globvenv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vidya\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activa

ValueError: Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.

In [ ]:
# mistral_test.py

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer and model...")

# Decide device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Use slow tokenizer (SentencePiece) to avoid fast-tokenizer issues
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,
)

# Load model on a single device, no device_map -> no accelerate needed
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
model.to(device)
model.eval()

print(f"Model loaded on {device}.")

def generate_bot_answer(question: str) -> str:
    prompt = (
        "You are a human player in a party game called 'Spot the Bot'. "
        "You must answer the question like a normal person, not like an AI. "
        "Be short, casual, and a bit messy if needed.\n\n"
        f"Question: {question}\n"
        "Your answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    full = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if full.startswith(prompt):
        full = full[len(prompt):]

    return full.strip()

if __name__ == "__main__":
    question = "What is your ideal weekend plan?"

    print("\n=== TEST RUN ===")
    print("Question (hard-coded in code):")
    print(question)

    answer = generate_bot_answer(question)

    print("\nGenerated answer (bot):")
    print(answer)
    print("================\n")



c:\My files\Programming\globvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer and model...


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]